# Indonesia Flood Early Warning System - Grid-Based Forecasting

Complete pipeline for flood risk prediction:
- **Grid System**: 3km × 3km cells across Indonesia
- **Risk Score**: 0-100 scale per grid cell
- **4 Comparable Models**: XGBoost, SARIMA, Prophet, LSTM
- **Forecast Horizon**: 3 months ahead
- **Risk Categories**: 4 levels (Rendah/Sedang/Tinggi/Sangat Tinggi)

## 1. Setup & Configuration

In [28]:
import sys
import os
from pathlib import Path

# Set paths explicitly
NOTEBOOK_DIR = Path(r"d:\Lomba\Iyref\TirtaIYREF\tirta-models\flood_history\src")
PROJECT_DIR = NOTEBOOK_DIR.parent  # flood_history
DATA_DIR = PROJECT_DIR / "data"
GRID_SIZE_KM = 5.0

# Add src to path
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

# Change working directory to flood_history (parent of src)
os.chdir(str(PROJECT_DIR))
print(f"Working directory: {os.getcwd()}")
print(f"Data path exists: {DATA_DIR.exists()}")
print(f"Data file exists: {(DATA_DIR / 'groundsource_2026.parquet').exists()}")

import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Import modules
from config import *
from data_preparation import GridBasedDataPreparation
from feature_engineering import FeatureEngineer
from models import ModelComparator
from forecasting import GridRiskForecaster, RiskCategorizer, get_risk_category_summary

print("\nAll imports successful")
print("\nConfiguration:")
print(f"  Grid size: {GRID_SIZE_KM:.0f}km × {GRID_SIZE_KM:.0f}km")
print(f"  Lookback: {LOOKBACK_WINDOW} months")
print(f"  Forecast horizon: {FORECAST_HORIZON} months")
print("  Models: XGBoost, SARIMA, Prophet, LSTM")

Working directory: d:\Lomba\Iyref\TirtaIYREF\tirta-models\flood_history
Data path exists: True
Data file exists: True

All imports successful

Configuration:
  Grid size: 5km × 5km
  Lookback: 12 months
  Forecast horizon: 3 months
  Models: XGBoost, SARIMA, Prophet, LSTM


## 2. Data Loading & Exploration

In [12]:
print("Loading flood data...")

data_prep = GridBasedDataPreparation(
    data_path=DATA_PATH,
    indonesia_bounds=INDONESIA_BOUNDS,
    cell_size_km=GRID_SIZE_KM,
    min_samples=DATA_CONFIG['min_samples_per_location']
)

print("\nStage 1: Loading raw data...")
data_prep.load_data()
print(f"  Loaded {len(data_prep.df):,} events")
print(f"  Columns: {data_prep.df.columns.tolist()}")

print("\nStage 2: Extracting coordinates...")
data_prep.extract_coordinates()
print("  Extracted coordinates from WKB geometry")
print(f"  Valid coords: {data_prep.df['lat'].notna().sum():,}")

print("\nStage 3: Filtering to Indonesia...")
result = data_prep.filter_indonesia()
print(f"  Filtered to {len(data_prep.df_indo):,} Indonesia events")

# Verify df_indo is not None
if data_prep.df_indo is None:
    print("ERROR: df_indo is None! Check filtering...")
else:
    print(f"  df_indo type: {type(data_prep.df_indo)}")
    
    # Ensure start_date is datetime
    data_prep.df_indo['start_date'] = pd.to_datetime(data_prep.df_indo['start_date'])
    
    print("\nData Summary:")
    print(f"  Date range: {data_prep.df_indo['start_date'].min().date()} to {data_prep.df_indo['start_date'].max().date()}")
    print(f"  Area range: {data_prep.df_indo['area_km2'].min():.1f} - {data_prep.df_indo['area_km2'].max():.1f} km²")
    print("  Geographic extent:")
    print(f"    Latitude:  [{data_prep.df_indo['lat'].min():.2f}, {data_prep.df_indo['lat'].max():.2f}]")
    print(f"    Longitude: [{data_prep.df_indo['lon'].min():.2f}, {data_prep.df_indo['lon'].max():.2f}]")

Loading flood data...

Stage 1: Loading raw data...
  Loaded 2,646,302 events
  Columns: ['uuid', 'area_km2', 'geometry', 'start_date', 'end_date']

Stage 2: Extracting coordinates...
  Extracted coordinates from WKB geometry
  Valid coords: 2,646,302

Stage 3: Filtering to Indonesia...
  Filtered to 370,156 Indonesia events
  df_indo type: <class 'pandas.DataFrame'>

Data Summary:
  Date range: 2000-04-17 to 2026-02-03
  Area range: 0.0 - 4993.6 km²
  Geographic extent:
    Latitude:  [-10.90, 6.00]
    Longitude: [95.04, 140.96]


## 3. Grid Generation & Assignment

In [15]:
print(f"Creating {GRID_SIZE_KM:.0f}km × {GRID_SIZE_KM:.0f}km grid...\n")

# Force complete module reload so the fresh class definition is used
import sys
import importlib

for mod in ['data_preparation', 'feature_engineering', 'models', 'forecasting', 'config']:
    if mod in sys.modules:
        importlib.reload(sys.modules[mod])

from config import *
from data_preparation import GridBasedDataPreparation

print("✓ All modules reloaded from disk\n")

# Recreate data preparation object with the fresh class definition
data_prep = GridBasedDataPreparation(
    data_path=DATA_PATH,
    indonesia_bounds=INDONESIA_BOUNDS,
    cell_size_km=GRID_SIZE_KM,
    min_samples=DATA_CONFIG['min_samples_per_location']
)

print("Stage 1: Loading raw data...")
data_prep.load_data()
print(f"✓ Loaded {len(data_prep.df):,} events")
print(f"  Columns: {data_prep.df.columns.tolist()}")

print("\nStage 2: Extracting coordinates...")
data_prep.extract_coordinates()
print("✓ Extracted coordinates from WKB geometry")
print(f"  Valid coords: {data_prep.df['lat'].notna().sum():,}")

print("\nStage 3: Filtering to Indonesia...")
data_prep.filter_indonesia()
print(f"✓ Filtered to {len(data_prep.df_indo):,} Indonesia events")

if data_prep.df_indo is None:
    raise ValueError("df_indo is still None after filtering")

# Ensure start_date is datetime
if 'start_date' in data_prep.df_indo.columns:
    data_prep.df_indo['start_date'] = pd.to_datetime(data_prep.df_indo['start_date'], errors='coerce')

print("\nData Summary:")
print(f"  Date range: {data_prep.df_indo['start_date'].min().date()} to {data_prep.df_indo['start_date'].max().date()}")
print(f"  Area range: {data_prep.df_indo['area_km2'].min():.1f} - {data_prep.df_indo['area_km2'].max():.1f} km²")
print("  Geographic extent:")
print(f"    Latitude:  [{data_prep.df_indo['lat'].min():.2f}, {data_prep.df_indo['lat'].max():.2f}]")
print(f"    Longitude: [{data_prep.df_indo['lon'].min():.2f}, {data_prep.df_indo['lon'].max():.2f}]")

# Create grid
print("\nStage 4: Generating grid...")
data_prep.grid_gen.create_grid()
print(f"✓ Grid generated with {len(data_prep.grid_gen.grid):,} cells")

# Assign events to grid
print("\nStage 4b: Assigning events to grid...")
data_prep.assign_to_grid()
print("✓ Events assigned to grid")
print(f"  Assigned events: {len(data_prep.df_indo):,}")

# Create monthly time series
print("\nStage 5: Creating monthly time series...")
df_timeseries = data_prep.create_monthly_timeseries()
print(f"✓ Time series created with {len(df_timeseries):,} records")

# Pad time series
print("\nStage 6: Padding time series...")
df_timeseries = data_prep.pad_timeseries(df_timeseries)
print("✓ Time series padded (zero-filled for missing months)")

# Generate risk labels
print("\nStage 7: Generating risk labels...")
df_timeseries = data_prep.generate_risk_labels(df_timeseries)
print("✓ Risk labels generated (0-100 score)")

print("\nGrid Statistics:")
print(f"  Total grid cells: {len(data_prep.grid_gen.grid):,}")
print(f"  Active cells (with events): {df_timeseries['grid_id'].nunique():,}")
print(f"  Risk score range: {df_timeseries['risk_score'].min():.1f} - {df_timeseries['risk_score'].max():.1f}")

Creating 5km × 5km grid...

✓ All modules reloaded from disk

Stage 1: Loading raw data...
✓ Loaded 2,646,302 events
  Columns: ['uuid', 'area_km2', 'geometry', 'start_date', 'end_date']

Stage 2: Extracting coordinates...
✓ Extracted coordinates from WKB geometry
  Valid coords: 2,646,302

Stage 3: Filtering to Indonesia...
✓ Filtered to 370,156 Indonesia events

Data Summary:
  Date range: 2000-04-17 to 2026-02-03
  Area range: 0.0 - 4993.6 km²
  Geographic extent:
    Latitude:  [-10.90, 6.00]
    Longitude: [95.04, 140.96]

Stage 4: Generating grid...
✓ Grid generated with 386,316 cells

Stage 4b: Assigning events to grid...
✓ Events assigned to grid
  Assigned events: 370,156

Stage 5: Creating monthly time series...
✓ Time series created with 166,282 records

Stage 6: Padding time series...
✓ Time series padded (zero-filled for missing months)

Stage 7: Generating risk labels...
✓ Risk labels generated (0-100 score)

Grid Statistics:
  Total grid cells: 386,316
  Active cells (wi

## 4. Feature Engineering

In [16]:
print("=" * 70)
print("DIAGNOSTICS: Feature Engineering Data Flow")
print("=" * 70)

print("\n1. TIMESERIES INPUT:")
print(f"   Total records in df_timeseries: {len(df_timeseries):,}")
print(f"   Unique grids: {df_timeseries['grid_id'].nunique():,}")
print(f"   Date range: {df_timeseries['date'].min()} to {df_timeseries['date'].max()}")
print(f"   Avg records per grid: {len(df_timeseries) / df_timeseries['grid_id'].nunique():.1f}")
print(f"   Min records per grid: {df_timeseries.groupby('grid_id').size().min()}")
print(f"   Max records per grid: {df_timeseries.groupby('grid_id').size().max()}")

# Create features with instrumentation
print("\n2. FEATURE ENGINEERING STAGES:")

fe = FeatureEngineer(FEATURE_CONFIG)
df_test = df_timeseries.copy()
df_test['date'] = pd.to_datetime(df_test['date'])
group_keys = ['grid_id'] if 'grid_id' in df_test.columns else ['lat', 'lon']
df_test = df_test.sort_values(group_keys + ['date']).reset_index(drop=True)

print(f"   Before any features: {len(df_test):,} rows")

# Add lagged features
df_test = fe.add_lagged_features(df_test)
print(f"   After lag features (with NaNs): {len(df_test):,} rows")

# Check NaN counts
nan_counts = df_test.isna().sum()
print("   NaN counts per column:")
for col, count in nan_counts[nan_counts > 0].items():
    print(f"     {col}: {count:,} ({count/len(df_test)*100:.1f}%)")

# Add other features
df_test = fe.add_cyclical_time_features(df_test)
df_test = fe.add_trend_features(df_test)
df_test = fe.add_rolling_statistics(df_test)

print(f"   Before dropna(): {len(df_test):,} rows")
print(f"   Total NaN values: {df_test.isna().sum().sum():,}")

# Show which rows would be dropped
df_before_drop = df_test.copy()
df_test = df_test.dropna()
print(f"   After dropna(): {len(df_test):,} rows ← DROPPED {len(df_before_drop) - len(df_test):,} rows!")

print("\n3. OUTPUT FEATURES:")
print(f"   Total features: {len(fe.feature_columns)}")
print(f"   Feature list: {fe.feature_columns}")
print(f"   Data points: {len(df_test):,}")
print(f"   Active grids: {df_test['grid_id'].nunique() if len(df_test) > 0 else 0:,}")

print("\n" + "=" * 70)

# Now run the actual feature engineering
print("\nCreating features for real...\n")

df_features, feature_cols = fe.create_features(df_timeseries)

print("Features created:")
print(f"  Total features: {len(feature_cols)}")
print(f"  Feature list: {feature_cols}")
print(f"  Data points: {len(df_features):,}")
print(f"  Active grids: {df_features['grid_id'].nunique():,}")

DIAGNOSTICS: Feature Engineering Data Flow

1. TIMESERIES INPUT:
   Total records in df_timeseries: 7,563,831
   Unique grids: 24,321
   Date range: 2000-04-01 00:00:00 to 2026-02-01 00:00:00
   Avg records per grid: 311.0
   Min records per grid: 311
   Max records per grid: 311

2. FEATURE ENGINEERING STAGES:
   Before any features: 7,563,831 rows
   After lag features (with NaNs): 7,563,831 rows
   NaN counts per column:
     risk_score_lag1: 24,321 (0.3%)
     risk_score_lag2: 48,642 (0.6%)
     risk_score_lag3: 72,963 (1.0%)
     risk_score_lag6: 145,926 (1.9%)
   Before dropna(): 7,563,831 rows
   Total NaN values: 291,852
   After dropna(): 7,417,905 rows ← DROPPED 145,926 rows!

3. OUTPUT FEATURES:
   Total features: 11
   Feature list: ['risk_score_lag1', 'risk_score_lag2', 'risk_score_lag3', 'risk_score_lag6', 'month_sin', 'month_cos', 'trend', 'risk_score_rolling_mean_3', 'risk_score_rolling_std_3', 'risk_score_rolling_mean_6', 'risk_score_rolling_std_6']
   Data points: 7,4

## 5. Train/Test Split

In [17]:
print("Splitting data (80/20 chronological split per grid)...\n")

# Prepare for modeling - get train/test splits
df_train, df_test = data_prep.prepare_for_modeling(
    df_features,
    lookback=LOOKBACK_WINDOW,
    test_split=0.2
)

# Extract features and targets
X_train = df_train[feature_cols].copy()
X_test = df_test[feature_cols].copy()
y_train = df_train['risk_score'].copy()
y_test = df_test['risk_score'].copy()

# Scale features
X_train_scaled, X_test_scaled = fe.scale_features(X_train, X_test)

print("✓ Data split complete:")
print(f"  Training samples: {len(X_train):,}")
print(f"  Test samples: {len(X_test):,}")
print(f"  Train/Test ratio: {len(X_train)}/{len(X_test)} = {len(X_train)/(len(X_train)+len(X_test))*100:.0f}%/{len(X_test)/(len(X_train)+len(X_test))*100:.0f}%")

Splitting data (80/20 chronological split per grid)...

✓ Data split complete:
  Training samples: 5,934,324
  Test samples: 1,483,581
  Train/Test ratio: 5934324/1483581 = 80%/20%


## 6. Model Training (4 Models)

In [18]:
print("Training risk forecasting models...\n")

# Ensure configs and models are loaded
from config import XGBOOST_CONFIG, SARIMA_CONFIG, PROPHET_CONFIG, LSTM_CONFIG

# FAST MODE: Only train XGBoost for quick pipeline validation
# Set FAST_MODE = False to train all 4 models (takes 30+ minutes)
FAST_MODE = True

if FAST_MODE:
    print("⚡ FAST MODE: Training XGBoost only (3-5 min)...\n")
    all_model_configs = {
        'xgboost': XGBOOST_CONFIG,
    }
else:
    print("📊 FULL MODE: Training all 4 models (30+ minutes)...\n")
    print("  - XGBoost: ~3 min")
    print("  - SARIMA: ~10 min")
    print("  - Prophet: ~5 min")
    print("  - LSTM (50 epochs): ~15 min\n")
    
    all_model_configs = {
        'xgboost': XGBOOST_CONFIG,
        'sarima': SARIMA_CONFIG,
        'prophet': PROPHET_CONFIG,
        'lstm': LSTM_CONFIG
    }

comparator = ModelComparator(all_model_configs)
comparator.init_models()

print(f"\nTraining {len(comparator.models)} model(s)...\n")

# --- KODE YANG DIUBAH ---
# Ubah DataFrame ke NumPy array agar XGBoost tidak menghafal nama fitur
X_train_array = X_train_scaled.to_numpy()
X_test_array = X_test_scaled.to_numpy()

# Masukkan variabel array ke dalam train_all
comparator.train_all(X_train_array, y_train)

print("\nEvaluating on test set...")
# Masukkan variabel array ke dalam predict_all
predictions = comparator.predict_all(X_test_array)
# -------------------------

results = comparator.evaluate_all(y_test, predictions)

print("\n✓ Model Evaluation Results:")
for model_name, metrics in results.items():
    print(f"\n{model_name.upper()}:")
    print(f"  MAE:  {metrics['mae']:.3f}")
    print(f"  RMSE: {metrics['rmse']:.3f}")
    print(f"  R²:   {metrics['r2']:.3f}")

if FAST_MODE:
    print("\n💡 Tip: Set FAST_MODE = False to train all 4 models for full comparison")

Training risk forecasting models...

⚡ FAST MODE: Training XGBoost only (3-5 min)...



Importing plotly failed. Interactive plots will not work.



Training 4 model(s)...



Training sarima failed: 'sarima'
Training prophet failed: 'prophet'
Training lstm failed: 'lstm'



Evaluating on test set...


Prediction with sarima failed: SARIMA not trained yet
Prediction with prophet failed: Prophet not trained yet
Prediction with lstm failed: LSTM not trained yet



✓ Model Evaluation Results:

XGBOOST:
  MAE:  0.335
  RMSE: 1.308
  R²:   0.986

💡 Tip: Set FAST_MODE = False to train all 4 models for full comparison


## 7. Forecasting (Grid-Based)

In [31]:
print("Creating forecasts for all grid cells with live progress...\n")

import time
import importlib
from tqdm.auto import tqdm
import forecasting as forecasting_module

# Reload forecasting module so latest code is used
importlib.reload(forecasting_module)
GridRiskForecaster = forecasting_module.GridRiskForecaster
RiskCategorizer = forecasting_module.RiskCategorizer

# Prepare full config dict for forecaster
full_config = {
    'RISK_CONFIG': RISK_CONFIG,
    'FEATURE_CONFIG': FEATURE_CONFIG,
}

# Create forecaster
risk_cat = RiskCategorizer(RISK_CONFIG)
forecaster = GridRiskForecaster(comparator, fe, full_config)

print("Forecasting with XGBoost (primary model)...")

# Build grid list (prefer active grids when flood_count exists)
if 'flood_count' in df_timeseries.columns:
    active_grid_ids = (
        df_timeseries.groupby('grid_id')['flood_count']
        .sum()
        .loc[lambda s: s >= 1]
        .index
        .tolist()
    )
    grid_ids = active_grid_ids
    print(f"Using active grids only (flood_count >= 1): {len(grid_ids):,} grids")
else:
    grid_ids = df_timeseries['grid_id'].unique().tolist()
    print(f"Using all grids: {len(grid_ids):,} grids")

all_forecasts = []
errors = []
start_ts = time.time()

progress = tqdm(grid_ids, desc='Forecasting grids', unit='grid')
for i, grid_id in enumerate(progress, start=1):
    try:
        res = forecaster.forecast_3months_grid(df_timeseries, int(grid_id), use_model='xgboost')
        if 'error' in res:
            errors.append((int(grid_id), res['error']))
        else:
            all_forecasts.append(res)
    except Exception as e:
        errors.append((int(grid_id), str(e)))

    # Update visible progress stats periodically
    if i % 200 == 0 or i == len(grid_ids):
        elapsed = time.time() - start_ts
        speed = i / elapsed if elapsed > 0 else 0
        eta = (len(grid_ids) - i) / speed if speed > 0 else 0
        progress.set_postfix({
            'ok': len(all_forecasts),
            'err': len(errors),
            'it/s': f"{speed:.2f}",
            'eta_s': f"{eta:.0f}"
        })
        print(
            f"Progress {i:,}/{len(grid_ids):,} | "
            f"success={len(all_forecasts):,} | error={len(errors):,} | "
            f"elapsed={elapsed/60:.1f}m | eta={eta/60:.1f}m"
        )

progress.close()

print(f"\nCompleted forecasts: {len(all_forecasts):,} successful, {len(errors):,} errors")

# Build DataFrame similar to forecaster.forecast_all_grids
output_data = []
for forecast in all_forecasts:
    first_pred = forecast['predictions'][0]
    output_data.append({
        'grid_id': forecast['grid_id'],
        'lat': forecast['lat'],
        'lon': forecast['lon'],
        'risk_score_month1': first_pred['risk_score'],
        'risk_score_month2': forecast['predictions'][1]['risk_score'],
        'risk_score_month3': forecast['predictions'][2]['risk_score'],
        'average_risk': forecast['average_risk'],
        'highest_risk_month': forecast['highest_risk_month'],
        'highest_risk_score': forecast['highest_risk_score'],
        'category_month1': first_pred['label'],
    })

forecast_df = pd.DataFrame(output_data)

print(f"\n✓ Forecasts generated for {len(forecast_df):,} active grid cells")
print("\nForecast columns:")
print(forecast_df.columns.tolist())

print("\nTop 10 highest risk grid cells:")
top_risk = forecast_df.nlargest(10, 'average_risk')[['grid_id', 'lat', 'lon', 'average_risk', 'category_month1']]
for idx, row in top_risk.iterrows():
    print(f"  Grid {int(row['grid_id']):>5} ({row['lat']:>7.2f}, {row['lon']:>8.2f}): Risk {row['average_risk']:>6.1f} | {row['category_month1']}")

if errors:
    print("\nSample errors (max 5):")
    for grid_id, err in errors[:5]:
        print(f"  Grid {grid_id}: {err}")

Creating forecasts for all grid cells with live progress...

Forecasting with XGBoost (primary model)...
Using active grids only (flood_count >= 1): 24,321 grids


Forecasting grids:   1%|          | 202/24321 [00:22<37:18, 10.77grid/s, ok=200, err=0, it/s=9.13, eta_s=2643]

Progress 200/24,321 | success=200 | error=0 | elapsed=0.4m | eta=44.1m


Forecasting grids:   2%|▏         | 402/24321 [00:40<36:17, 10.98grid/s, ok=400, err=0, it/s=9.83, eta_s=2432]  

Progress 400/24,321 | success=400 | error=0 | elapsed=0.7m | eta=40.5m


...

Progress 24,000/24,321 | success=24,000 | error=0 | elapsed=43.6m | eta=0.6m


Forecasting grids: 100%|█████████▉| 24201/24321 [44:01<00:17,  6.71grid/s, ok=24200, err=0, it/s=9.16, eta_s=13]

Progress 24,200/24,321 | success=24,200 | error=0 | elapsed=44.0m | eta=0.2m


Forecasting grids: 100%|██████████| 24321/24321 [44:19<00:00,  9.15grid/s, ok=24321, err=0, it/s=9.15, eta_s=0] 


Progress 24,321/24,321 | success=24,321 | error=0 | elapsed=44.3m | eta=0.0m

Completed forecasts: 24,321 successful, 0 errors

✓ Forecasts generated for 24,321 active grid cells

Forecast columns:
['grid_id', 'lat', 'lon', 'risk_score_month1', 'risk_score_month2', 'risk_score_month3', 'average_risk', 'highest_risk_month', 'highest_risk_score', 'category_month1']

Top 10 highest risk grid cells:
  Grid 64801 (  -8.14,   113.70): Risk   54.3 | Rawan
  Grid 86242 (  -7.21,   112.77): Risk   54.2 | Rawan
  Grid 88278 (  -7.12,   112.42): Risk   54.2 | Rawan
  Grid 96369 (  -6.74,   108.58): Risk   54.2 | Rawan
  Grid 110641 (  -6.13,   106.94): Risk   54.2 | Rawan
  Grid 112663 (  -6.02,   105.98): Risk   54.2 | Rawan
  Grid 173153 (  -3.36,   114.61): Risk   54.2 | Rawan
  Grid 221897 (  -1.19,   100.59): Risk   54.2 | Rawan
  Grid 229048 (  -0.89,   100.41): Risk   54.2 | Rawan
  Grid 231090 (  -0.78,   100.33): Risk   54.2 | Rawan


## 8. Risk Categories Reference

In [25]:
print("Risk Categories & Definitions:\n")

categories = get_risk_category_summary()

# Fallback colors for backward compatibility if old module output lacks `color`
default_colors = {
    'Rendah': '#2ca02c',
    'Sedang': '#ffd700',
    'Tinggi': '#ff7f0e',
    'Sangat Tinggi': '#d62728',
}

print(f"{'Level':<15} {'English':<20} {'Indonesian':<20} {'Score Range':<15} {'Color':<10}")
print("-" * 80)

for cat_name, cat_info in categories.items():
    color = cat_info.get('color', default_colors.get(cat_name, '#999999'))
    print(f"{cat_name:<15} {cat_info.get('label', ''):<20} {cat_info.get('label_id', cat_name):<20} {str(cat_info.get('range', '')):<15} {color:<10}")

print("\nRisk Score Interpretation:")
print("  0-25:   Low risk, minimal flood concern")
print("  25-50:  Moderate risk, monitoring recommended")
print("  50-75:  High risk, preparedness needed")
print("  75-100: Very high risk, active warning issued")

Risk Categories & Definitions:

Level           English              Indonesian           Score Range     Color     
--------------------------------------------------------------------------------
Rendah          Tidak Rawan          Rendah               0-25            #2ca02c   
Sedang          Cukup Rawan          Sedang               25-50           #ffd700   
Tinggi          Rawan                Tinggi               50-75           #ff7f0e   
Sangat Tinggi   Sangat Rawan         Sangat Tinggi        75-100          #d62728   

Risk Score Interpretation:
  0-25:   Low risk, minimal flood concern
  25-50:  Moderate risk, monitoring recommended
  50-75:  High risk, preparedness needed
  75-100: Very high risk, active warning issued


## 9. Model Comparison on Sample Grid

In [26]:
print("Comparing all 4 models on a sample grid cell...\n")

# Select a high-risk grid
sample_grid = forecast_df.nlargest(1, 'average_risk')['grid_id'].values[0]
sample_grid = int(sample_grid)

print(f"Sample Grid: {sample_grid}")

# Compare models
comparison = forecaster.compare_models_grid(df_timeseries, sample_grid)

if 'models' in comparison:
    print("\nPer-Model Risk Forecasts (3-Month Average):")
    for model_name, result in comparison['models'].items():
        print(f"  {model_name:<12} Average Risk: {result['average_risk']:.1f}")
    
    print("\nComparison Summary:")
    summary = comparison['summary']
    print(f"  Average across models: {summary['avg_risk_across_models']:.1f}")
    print(f"  Minimum risk: {summary['min_risk']:.1f}")
    print(f"  Maximum risk: {summary['max_risk']:.1f}")
    print(f"  Risk range: {summary['max_risk'] - summary['min_risk']:.1f}")

Prediction failed for grid 229293: SARIMA not trained yet
Prediction failed for grid 229293: Prophet not trained yet


Comparing all 4 models on a sample grid cell...

Sample Grid: 229293


Prediction failed for grid 229293: LSTM not trained yet



Per-Model Risk Forecasts (3-Month Average):
  xgboost      Average Risk: 59.9

Comparison Summary:
  Average across models: 59.9
  Minimum risk: 59.9
  Maximum risk: 59.9
  Risk range: 0.0


## 10. Summary & Export

In [32]:
import json
import pickle
from datetime import datetime
from pathlib import Path

print("\n" + "="*70)
print("PIPELINE COMPLETE - SUMMARY")
print("="*70)

print("\n📊 SYSTEM STATISTICS:")
print(f"  Grid cells processed: {forecast_df['grid_id'].nunique():,}")
print(f"  Forecast records: {len(forecast_df):,}")
print(f"  Models trained: {len(comparator.models)}")
print(f"  Forecast horizon: {FORECAST_HORIZON} months")
print(f"  Feature count: {len(feature_cols)}")

print("\n🎯 RISK DISTRIBUTION:")
risk_counts = forecast_df['category_month1'].value_counts()
for risk, count in risk_counts.items():
    pct = count / len(forecast_df) * 100
    print(f"  {risk:<12} {count:>5} cells ({pct:>5.1f}%)")

# ============================================================
# SAVE TRAINED MODELS + FORECAST OUTPUT (JSON)
# ============================================================
output_root = Path("output")
models_dir = output_root / "models"
exports_dir = output_root / "exports"
models_dir.mkdir(parents=True, exist_ok=True)
exports_dir.mkdir(parents=True, exist_ok=True)

saved_models = {}
for model_name, model_obj in comparator.models.items():
    try:
        # Most wrappers store the trained estimator in `.model`
        trained = getattr(model_obj, 'model', None)
        if trained is None:
            continue

        if model_name == 'xgboost' and hasattr(trained, 'save_model'):
            model_path = models_dir / f"{model_name}_model.json"
            trained.save_model(str(model_path))
        elif model_name == 'lstm' and hasattr(trained, 'save'):
            model_path = models_dir / f"{model_name}_model.keras"
            trained.save(str(model_path))
        else:
            model_path = models_dir / f"{model_name}_model.pkl"
            with open(model_path, 'wb') as f:
                pickle.dump(trained, f)

        saved_models[model_name] = str(model_path)
    except Exception as e:
        print(f"  ⚠️ Failed saving model {model_name}: {e}")

# Save model metrics/evaluation JSON
metrics_payload = {
    'generated_at': datetime.utcnow().isoformat() + 'Z',
    'models': list(comparator.models.keys()),
    'saved_models': saved_models,
    'evaluation': results if 'results' in globals() else {},
    'feature_columns': feature_cols,
    'forecast_horizon': FORECAST_HORIZON,
}

metrics_path = exports_dir / "model_metrics.json"
with open(metrics_path, 'w', encoding='utf-8') as f:
    json.dump(metrics_payload, f, ensure_ascii=False, indent=2)

# Save full forecast JSON
forecast_payload = {
    'generated_at': datetime.utcnow().isoformat() + 'Z',
    'grid_size_km': GRID_SIZE_KM,
    'forecast_horizon': FORECAST_HORIZON,
    'record_count': int(len(forecast_df)),
    'columns': list(forecast_df.columns),
    'data': forecast_df.to_dict(orient='records')
}

forecast_path = exports_dir / "forecast_output.json"
with open(forecast_path, 'w', encoding='utf-8') as f:
    json.dump(forecast_payload, f, ensure_ascii=False, indent=2)

print("\n💾 SAVED ARTIFACTS:")
for name, path in saved_models.items():
    print(f"  ✓ Model {name:<10} -> {path}")
print(f"  ✓ Metrics JSON  -> {metrics_path}")
print(f"  ✓ Forecast JSON -> {forecast_path}")

print("\n✅ READY FOR DEPLOYMENT:")
print("  ✓ Grid-based forecasting operational")
print("  ✓ Risk scores calculated (0-100)")
print("  ✓ Risk categories assigned")
print("  ✓ Trained model(s) saved")
print("  ✓ Forecast output exported as JSON")
print("\n💡 NEXT STEPS:")
print("  1. Load saved model from output/models")
print("  2. Consume forecast JSON from output/exports/forecast_output.json")
print("  3. Build API endpoint for realtime inference")

print("\n" + "="*70)


PIPELINE COMPLETE - SUMMARY

📊 SYSTEM STATISTICS:
  Grid cells processed: 24,321
  Forecast records: 24,321
  Models trained: 4
  Forecast horizon: 3 months
  Feature count: 11

🎯 RISK DISTRIBUTION:
  Tidak Rawan  23354 cells ( 96.0%)
  Cukup Rawan    665 cells (  2.7%)
  Rawan          302 cells (  1.2%)

💾 SAVED ARTIFACTS:
  ✓ Model xgboost    -> output\models\xgboost_model.json
  ✓ Metrics JSON  -> output\exports\model_metrics.json
  ✓ Forecast JSON -> output\exports\forecast_output.json

✅ READY FOR DEPLOYMENT:
  ✓ Grid-based forecasting operational
  ✓ Risk scores calculated (0-100)
  ✓ Risk categories assigned
  ✓ Trained model(s) saved
  ✓ Forecast output exported as JSON

💡 NEXT STEPS:
  1. Load saved model from output/models
  2. Consume forecast JSON from output/exports/forecast_output.json
  3. Build API endpoint for realtime inference

